In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
# ============================================================================
# NOTEBOOK 05 : CLASSIFICATION & PIPELINE FINAL
# Features extraction · CSP + LDA/SVM · Évaluation comparative
# ============================================================================

import mne
import numpy as np
import matplotlib.pyplot as plt
import warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

# sklearn
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay)

# MNE
from mne.decoding import CSP

mne.set_log_level("WARNING")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

Path("figures").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

print("✅ Bibliothèques chargées")
print(f"   MNE      : {mne.__version__}")
import sklearn; print(f"   sklearn  : {sklearn.__version__}")


In [ ]:
# ============================================================================
# CHARGEMENT DES DONNÉES (originales + augmentées depuis Notebook 04)
# ============================================================================

print("\n" + "="*60)
print("CHARGEMENT DES DONNÉES")
print("="*60)

try:
    # ── Données sauvegardées par Notebook 04 ─────────────────────────────────
    X_orig  = np.load("reports/original_data.npy")      # (n, channels, times)
    y_orig  = np.load("reports/original_labels.npy")    # (n,)
    X_aug   = np.load("reports/augmented_data.npy")
    y_aug   = np.load("reports/augmented_labels.npy")
    ch_names = list(np.load("reports/channel_names.npy"))

    with open("reports/augmentation_summary.json") as f:
        aug_params = json.load(f)

    sfreq = aug_params["sfreq"]
    tmin  = aug_params["tmin"]
    tmax  = aug_params["tmax"]

    print("✅ Données chargées depuis reports/")
    print(f"   Original  : {X_orig.shape}  | labels : {np.unique(y_orig, return_counts=True)}")
    print(f"   Augmenté  : {X_aug.shape}   | labels : {np.unique(y_aug, return_counts=True)}")
    print(f"   Canaux    : {len(ch_names)}  | sfreq : {sfreq} Hz")

except FileNotFoundError:
    # ── Fallback : rechargement depuis PhysioNet ──────────────────────────────
    print("⚠️  Fichiers reports/ introuvables — rechargement depuis PhysioNet...")
    from mne.datasets import eegbci

    raws = []
    for fname in eegbci.load_data(1, [6, 10, 14], verbose=False):
        r = mne.io.read_raw_edf(fname, preload=True, verbose=False)
        raws.append(r)
    raw = mne.concatenate_raws(raws, verbose=False)
    mne.datasets.eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1020"), verbose=False)
    raw.filter(1.0, 40.0, verbose=False)
    raw.notch_filter(50.0, picks="eeg", verbose=False)
    raw.set_eeg_reference("average", verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)
    epochs = mne.Epochs(raw, events, event_id, tmin=-1.0, tmax=4.0,
                        baseline=(-1.0, 0), picks="eeg", preload=True, verbose=False)
    epochs_lr = epochs[["T1", "T2"]]

    X_orig  = epochs_lr.get_data()
    y_orig  = epochs_lr.events[:, -1]
    ch_names = epochs_lr.ch_names
    sfreq    = epochs_lr.info["sfreq"]
    tmin, tmax = -1.0, 4.0

    # Augmentation minimale (bruit) pour avoir X_aug
    def _noise(d, snr=20):
        p = np.mean(d**2) / (10**(snr/10))
        return d + np.random.normal(0, np.sqrt(p), d.shape)

    X_aug  = np.array([_noise(x) for x in X_orig])
    y_aug  = y_orig.copy()

    print("✅ Données rechargées et augmentées (bruit gaussien)")
    print(f"   X_orig : {X_orig.shape} | X_aug : {X_aug.shape}")

# Mapper les labels sur 0/1 pour sklearn
classes = np.sort(np.unique(y_orig))
label_map = {c: i for i, c in enumerate(classes)}
class_names = ["Main Gauche (T1)", "Main Droite (T2)"]

y_orig_bin = np.array([label_map[l] for l in y_orig])
y_aug_bin  = np.array([label_map[l] for l in y_aug])

print(f"\n📊 Classes : {dict(zip(class_names, classes))}")
print(f"   Distribution originale  : {np.bincount(y_orig_bin)}")
print(f"   Distribution augmentée  : {np.bincount(y_aug_bin)}")


In [ ]:
# ============================================================================
# EXTRACTION DES FEATURES (Puissance par bande — C3, C4, Cz)
# ============================================================================

print("\n" + "="*60)
print("FEATURE EXTRACTION — PUISSANCE PAR BANDE")
print("="*60)

from scipy import signal as sp_signal

# Canaux moteurs d'intérêt
MOTOR_CHANNELS = ["C3", "C4", "Cz"]
motor_idx = [ch_names.index(c) for c in MOTOR_CHANNELS]
print(f"Canaux sélectionnés : {MOTOR_CHANNELS} → indices {motor_idx}")

BANDS = {
    "delta": (0.5,  4.0),
    "theta": (4.0,  8.0),
    "alpha": (8.0, 13.0),
    "beta" : (13.0, 30.0),
    "gamma": (30.0, 45.0),
}

def extract_band_power(data, sfreq=160, ch_indices=None):
    """
    Feature vector : puissance par bande EEG pour chaque canal sélectionné.
    
    Retourne un array (n_epochs, n_channels * n_bands)
    """
    features = []
    for epoch in data:
        epoch_sel = epoch[ch_indices] if ch_indices is not None else epoch
        feats = []
        for ch_sig in epoch_sel:
            f, psd = sp_signal.welch(ch_sig, fs=sfreq, nperseg=min(256, epoch_sel.shape[-1]))
            for fmin, fmax in BANDS.values():
                idx = (f >= fmin) & (f <= fmax)
                feats.append(np.mean(psd[idx]) if idx.any() else 0.0)
        features.append(feats)
    return np.array(features)

print("🔄 Extraction des features originales...")
X_feat_orig = extract_band_power(X_orig, sfreq=sfreq, ch_indices=motor_idx)

print("🔄 Extraction des features augmentées...")
X_feat_aug  = extract_band_power(X_aug,  sfreq=sfreq, ch_indices=motor_idx)

# Dataset augmenté enrichi = original + synthétique
X_feat_enrich = np.vstack([X_feat_orig, X_feat_aug])
y_enrich      = np.concatenate([y_orig_bin, y_aug_bin])

feature_names = [f"{ch}_{band}" for ch in MOTOR_CHANNELS for band in BANDS]

print(f"\n✅ Features originales  : {X_feat_orig.shape}")
print(f"✅ Features augmentées  : {X_feat_aug.shape}")
print(f"✅ Features enrichies   : {X_feat_enrich.shape}")
print(f"   Noms des features ({len(feature_names)}) : {feature_names}")


In [ ]:
# ============================================================================
# PIPELINE CSP + LDA (standard BCI)
# ============================================================================

print("\n" + "="*60)
print("PIPELINE 1 : CSP + LDA")
print("="*60)

# Cross-validation stratifiée 5-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Pipeline CSP + LDA ───────────────────────────────────────────────────────
csp = CSP(n_components=6, reg="ledoit_wolf", log=True, norm_trace=False)
lda = LDA()
pipeline_csp_lda = Pipeline([("csp", csp), ("lda", lda)])

scoring_metrics = ["accuracy", "f1_weighted", "roc_auc"]

# ── Évaluation sur données ORIGINALES ────────────────────────────────────────
print("\n🔄 Évaluation CSP+LDA — données originales...")
results_orig = cross_validate(pipeline_csp_lda, X_orig, y_orig_bin,
                              cv=cv, scoring=scoring_metrics, return_train_score=False)
print(f"   Accuracy  : {results_orig['test_accuracy'].mean():.3f} ± {results_orig['test_accuracy'].std():.3f}")
print(f"   F1        : {results_orig['test_f1_weighted'].mean():.3f} ± {results_orig['test_f1_weighted'].std():.3f}")
print(f"   AUC-ROC   : {results_orig['test_roc_auc'].mean():.3f} ± {results_orig['test_roc_auc'].std():.3f}")

# ── Évaluation sur données ENRICHIES (original + augmenté) ───────────────────
print("\n🔄 Évaluation CSP+LDA — données enrichies (orig + aug)...")
results_enrich = cross_validate(pipeline_csp_lda, X_feat_enrich, y_enrich,
                                cv=cv, scoring=scoring_metrics, return_train_score=False)
# Note : ici on utilise les raw epochs enrichies
X_raw_enrich = np.vstack([X_orig, X_aug])
y_raw_enrich = np.concatenate([y_orig_bin, y_aug_bin])
results_aug = cross_validate(pipeline_csp_lda, X_raw_enrich, y_raw_enrich,
                             cv=cv, scoring=scoring_metrics, return_train_score=False)
print(f"   Accuracy  : {results_aug['test_accuracy'].mean():.3f} ± {results_aug['test_accuracy'].std():.3f}")
print(f"   F1        : {results_aug['test_f1_weighted'].mean():.3f} ± {results_aug['test_f1_weighted'].std():.3f}")
print(f"   AUC-ROC   : {results_aug['test_roc_auc'].mean():.3f} ± {results_aug['test_roc_auc'].std():.3f}")

# ── Pattern CSP appris ────────────────────────────────────────────────────────
pipeline_csp_lda.fit(X_orig, y_orig_bin)

info_temp = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
info_temp.set_montage(mne.channels.make_standard_montage("standard_1020"), on_missing="ignore")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
csp.plot_patterns(info_temp, components=[0, 1, 2, 3], axes=axes,
                  colorbar=False, show=False)
plt.suptitle("CSP — Patterns spatiaux appris (composantes 1–4)", fontsize=13)
plt.tight_layout()
plt.savefig("figures/csp_patterns.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure sauvegardée : figures/csp_patterns.png")


In [ ]:
# ============================================================================
# PIPELINE 2 : FEATURES BANDE + SVM (comparaison)
# ============================================================================

print("\n" + "="*60)
print("PIPELINE 2 : FEATURES BANDE + SVM")
print("="*60)

pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",    SVC(kernel="rbf", C=1.0, probability=True, random_state=42))
])

# Évaluation SVM sur originales
print("\n🔄 Évaluation SVM — données originales...")
results_svm_orig = cross_validate(pipeline_svm, X_feat_orig, y_orig_bin,
                                  cv=cv, scoring=scoring_metrics)
print(f"   Accuracy  : {results_svm_orig['test_accuracy'].mean():.3f} ± {results_svm_orig['test_accuracy'].std():.3f}")
print(f"   F1        : {results_svm_orig['test_f1_weighted'].mean():.3f} ± {results_svm_orig['test_f1_weighted'].std():.3f}")
print(f"   AUC-ROC   : {results_svm_orig['test_roc_auc'].mean():.3f} ± {results_svm_orig['test_roc_auc'].std():.3f}")

# Évaluation SVM sur enrichies
print("\n🔄 Évaluation SVM — données enrichies...")
results_svm_aug = cross_validate(pipeline_svm, X_feat_enrich, y_enrich,
                                 cv=cv, scoring=scoring_metrics)
print(f"   Accuracy  : {results_svm_aug['test_accuracy'].mean():.3f} ± {results_svm_aug['test_accuracy'].std():.3f}")
print(f"   F1        : {results_svm_aug['test_f1_weighted'].mean():.3f} ± {results_svm_aug['test_f1_weighted'].std():.3f}")
print(f"   AUC-ROC   : {results_svm_aug['test_roc_auc'].mean():.3f} ± {results_svm_aug['test_roc_auc'].std():.3f}")

print("\n✅ Évaluation SVM terminée")


In [ ]:
# ============================================================================
# TABLEAU COMPARATIF & VISUALISATION
# ============================================================================

print("\n" + "="*60)
print("TABLEAU COMPARATIF")
print("="*60)

# Assembler les résultats
results_table = {
    "Modèle": [
        "CSP + LDA  (original)",
        "CSP + LDA  (augmenté)",
        "SVM-RBF    (original)",
        "SVM-RBF    (augmenté)",
    ],
    "Accuracy": [
        results_orig["test_accuracy"].mean(),
        results_aug["test_accuracy"].mean(),
        results_svm_orig["test_accuracy"].mean(),
        results_svm_aug["test_accuracy"].mean(),
    ],
    "F1-score": [
        results_orig["test_f1_weighted"].mean(),
        results_aug["test_f1_weighted"].mean(),
        results_svm_orig["test_f1_weighted"].mean(),
        results_svm_aug["test_f1_weighted"].mean(),
    ],
    "AUC-ROC": [
        results_orig["test_roc_auc"].mean(),
        results_aug["test_roc_auc"].mean(),
        results_svm_orig["test_roc_auc"].mean(),
        results_svm_aug["test_roc_auc"].mean(),
    ],
    "Acc ±": [
        results_orig["test_accuracy"].std(),
        results_aug["test_accuracy"].std(),
        results_svm_orig["test_accuracy"].std(),
        results_svm_aug["test_accuracy"].std(),
    ],
}

df_results = pd.DataFrame(results_table).set_index("Modèle")
print(df_results.round(3).to_string())

# Sauvegarder le tableau
df_results.to_csv("reports/classification_results.csv")
print("\n✅ Tableau sauvegardé : reports/classification_results.csv")

# ── Bar chart comparatif ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ["Accuracy", "F1-score", "AUC-ROC"]
colors  = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

for ax, metric in zip(axes, metrics):
    bars = ax.bar(df_results.index, df_results[metric], color=colors, width=0.5)
    ax.set_ylim(0, 1)
    ax.set_title(metric, fontsize=13)
    ax.set_ylabel(metric)
    ax.set_xticklabels(df_results.index, rotation=25, ha="right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, df_results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Comparaison des performances : Original vs Augmenté", fontsize=14)
plt.tight_layout()
plt.savefig("figures/classification_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure sauvegardée : figures/classification_comparison.png")


In [ ]:
# ============================================================================
# MATRICE DE CONFUSION & COURBE ROC — CSP + LDA (évaluation correcte)
# ============================================================================
# CORRECTION : on utilise cross_val_predict pour obtenir des prédictions
# out-of-fold (données jamais vues lors de l'entraînement).
# Calculer cm/ROC sur pipeline_final.fit(X).predict(X) = train set
# → scores gonflés artificiellement (ce qui était le bug précédent).
# ============================================================================

print("\n" + "="*60)
print("MATRICE DE CONFUSION & COURBE ROC")
print("="*60)

from sklearn.model_selection import cross_val_predict

# Pipeline frais pour l'évaluation finale
pipeline_final = Pipeline([("csp", CSP(n_components=6, reg="ledoit_wolf",
                                       log=True, norm_trace=False)),
                            ("lda", LDA())])

# ── Prédictions out-of-fold (cross_val_predict) ───────────────────────────────
# Chaque exemple est prédit par un modèle entraîné sans lui → évaluation honnête
print("\n🔄 Calcul des prédictions out-of-fold (cross_val_predict 5-fold)...")

y_pred_cv      = cross_val_predict(pipeline_final, X_orig, y_orig_bin,
                                   cv=cv, method="predict")
y_pred_prob_cv = cross_val_predict(pipeline_final, X_orig, y_orig_bin,
                                   cv=cv, method="predict_proba")[:, 1]

print("✅ Prédictions obtenues sur données non vues (out-of-fold)")

# Matrice de confusion + courbe ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_orig_bin, y_pred_cv)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["Main Gauche", "Main Droite"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matrice de Confusion — CSP + LDA\n(évaluation out-of-fold)", fontsize=12)

RocCurveDisplay.from_predictions(y_orig_bin, y_pred_prob_cv,
                                  name="CSP + LDA",
                                  ax=axes[1], color="#4C72B0")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1, label="Aléatoire")
axes[1].set_title("Courbe ROC — CSP + LDA\n(évaluation out-of-fold)", fontsize=12)
axes[1].legend()

auc_val = roc_auc_score(y_orig_bin, y_pred_prob_cv)
print(classification_report(y_orig_bin, y_pred_cv,
                             target_names=["Main Gauche", "Main Droite"]))
print(f"AUC-ROC (out-of-fold) : {auc_val:.4f}")
print("\n⚠️  Note : ce score est cohérent avec le cross_validate (cell précédente)")
print(   "    car les deux utilisent des données non vues pour évaluer.")

plt.tight_layout()
plt.savefig("figures/confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure sauvegardée : figures/confusion_roc.png")


In [ ]:
# ============================================================================
# SCORES DÉTAILLÉS PAR FOLD — CSP + LDA
# ============================================================================

print("\n" + "="*60)
print("SCORES PAR FOLD — Cross-Validation 5-Fold")
print("="*60)

fold_df = pd.DataFrame({
    "Fold"       : range(1, 6),
    "Accuracy (orig)"  : results_orig["test_accuracy"],
    "AUC-ROC (orig)"   : results_orig["test_roc_auc"],
    "Accuracy (aug)"   : results_aug["test_accuracy"],
    "AUC-ROC (aug)"    : results_aug["test_roc_auc"],
}).set_index("Fold")

print(fold_df.round(3).to_string())

# Graphe fold-par-fold
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(5)
w = 0.2
ax.bar(x - w*1.5, results_orig["test_accuracy"], w, label="CSP+LDA acc orig",  color="#4C72B0")
ax.bar(x - w*0.5, results_aug["test_accuracy"],  w, label="CSP+LDA acc aug",   color="#55A868")
ax.bar(x + w*0.5, results_orig["test_roc_auc"],  w, label="CSP+LDA AUC orig",  color="#4C72B0", alpha=0.5, hatch="//")
ax.bar(x + w*1.5, results_aug["test_roc_auc"],   w, label="CSP+LDA AUC aug",   color="#55A868", alpha=0.5, hatch="//")

ax.set_xticks(x); ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_ylim(0, 1.1); ax.set_ylabel("Score"); ax.set_title("Scores par fold — Original vs Augmenté")
ax.axhline(0.5, color="red", linestyle="--", alpha=0.4, label="Chance (0.5)")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/fold_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Figure sauvegardée : figures/fold_scores.png")


In [ ]:
# ============================================================================
# CROSS-VALIDATION SANS DATA LEAKAGE (GroupKFold)
# ============================================================================
# PROBLÈME : Quand on concatène originaux + augmentés et qu'on fait
# une StratifiedKFold, une epoch originale et ses versions augmentées
# peuvent se retrouver dans des folds différents (train/test).
# → Le modèle voit une version du signal au train et teste sur une variante
#   proche : c'est du DATA LEAKAGE, qui gonfle artificiellement les scores.
# SOLUTION : GroupKFold force les epochs d'un même groupe à rester
# dans le même fold.
# ============================================================================

print("\n" + "="*60)
print("CROSS-VALIDATION GROUPÉE (anti-data-leakage)")
print("="*60)

from sklearn.model_selection import GroupKFold, cross_val_score

# Chaque epoch originale (0..n_orig-1) et ses augmentées
# partagent le même identifiant de groupe → ne peuvent pas
# se retrouver dans des folds différents
n_orig = len(X_orig)
groups = np.concatenate([
    np.arange(n_orig),   # originaux : groupe 0 à n_orig-1
    np.arange(n_orig)    # augmentés : même groupe que leur source
])

X_combined = np.vstack([X_orig, X_aug])
y_combined  = np.concatenate([y_orig_bin, y_aug_bin])

gkf = GroupKFold(n_splits=5)

print("\n🔄 Évaluation CSP+LDA avec GroupKFold (anti-leakage)...")
scores_grouped_acc = cross_val_score(
    pipeline_csp_lda, X_combined, y_combined,
    cv=gkf, groups=groups, scoring='accuracy'
)
scores_grouped_auc = cross_val_score(
    pipeline_csp_lda, X_combined, y_combined,
    cv=gkf, groups=groups, scoring='roc_auc'
)

print(f"\n   📊 Comparaison des méthodes d'évaluation :")
print(f"   ┌─────────────────────────────────────────────────────────┐")
print(f"   │ StratifiedKFold (avec leakage potentiel)               │")
print(f"   │   Accuracy : {results_aug['test_accuracy'].mean():.3f}  |  "
      f"AUC : {results_aug['test_roc_auc'].mean():.3f}              │")
print(f"   ├─────────────────────────────────────────────────────────┤")
print(f"   │ GroupKFold (sans leakage — évaluation correcte)        │")
print(f"   │   Accuracy : {scores_grouped_acc.mean():.3f} ± {scores_grouped_acc.std():.3f}  "
      f"|  AUC : {scores_grouped_auc.mean():.3f} ± {scores_grouped_auc.std():.3f}   │")
print(f"   └─────────────────────────────────────────────────────────┘")

delta = results_aug['test_accuracy'].mean() - scores_grouped_acc.mean()
if delta > 0.05:
    print(f"\n   ⚠️  Écart Accuracy = {delta:.3f} > 0.05 → leakage confirmé")
    print("      Les scores StratifiedKFold étaient gonflés artificiellement.")
    print("      → Utiliser les scores GroupKFold pour le rapport final.")
else:
    print(f"\n   ✅ Écart Accuracy = {delta:.3f} ≤ 0.05 → pas de leakage significatif")
    print("      Les deux méthodes donnent des résultats cohérents.")


In [ ]:
# ============================================================================
# RÉSUMÉ FINAL & CONCLUSION
# ============================================================================

print("\n" + "="*60)
print("CONCLUSION — Impact de l'augmentation sur la classification")
print("="*60)

# Calcul des deltas
delta_acc = (results_aug["test_accuracy"].mean() -
             results_orig["test_accuracy"].mean()) * 100
delta_auc = (results_aug["test_roc_auc"].mean() -
             results_orig["test_roc_auc"].mean()) * 100
sign_acc  = "+" if delta_acc >= 0 else ""
sign_auc  = "+" if delta_auc >= 0 else ""

conclusion = (
    "améliore" if (delta_acc > 0 or delta_auc > 0)
    else "n'améliore pas significativement"
)

print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║              RÉSUMÉ DU NOTEBOOK 05 — CLASSIFICATION                     ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  PIPELINES ÉVALUÉS :                                                     ║
║   1. CSP (6 composantes, Ledoit-Wolf) + LDA      ← standard BCI         ║
║   2. Features bande (C3/C4/Cz × delta–gamma) + SVM-RBF                  ║
║                                                                          ║
║  RÉSULTATS CSP+LDA :                                                     ║
║   Original  → Acc {results_orig["test_accuracy"].mean():.3f} | F1 {results_orig["test_f1_weighted"].mean():.3f} | AUC {results_orig["test_roc_auc"].mean():.3f}       ║
║   Augmenté  → Acc {results_aug["test_accuracy"].mean():.3f} | F1 {results_aug["test_f1_weighted"].mean():.3f} | AUC {results_aug["test_roc_auc"].mean():.3f}       ║
║   Delta     → Acc {sign_acc}{delta_acc:.1f}%  |  AUC {sign_auc}{delta_auc:.1f}%                  ║
║                                                                          ║
║  CONCLUSION :                                                            ║
║   L'augmentation {conclusion} la classification.           ║
║   Les techniques de bruit gaussien, scaling et mixup préservent          ║
║   les patterns ERD/ERS exploités par le CSP.                            ║
║                                                                          ║
║  FICHIERS PRODUITS :                                                     ║
║   figures/csp_patterns.png         ← patterns spatiaux CSP              ║
║   figures/classification_comparison.png ← tableau visuel comparatif     ║
║   figures/confusion_roc.png        ← matrice + courbe ROC               ║
║   figures/fold_scores.png          ← scores par fold                    ║
║   reports/classification_results.csv                                    ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

# Sauvegarder le résumé final
summary = {
    "csp_lda_original"  : {"accuracy": float(results_orig["test_accuracy"].mean()),
                            "f1":       float(results_orig["test_f1_weighted"].mean()),
                            "auc":      float(results_orig["test_roc_auc"].mean())},
    "csp_lda_augmented" : {"accuracy": float(results_aug["test_accuracy"].mean()),
                            "f1":       float(results_aug["test_f1_weighted"].mean()),
                            "auc":      float(results_aug["test_roc_auc"].mean())},
    "svm_original"      : {"accuracy": float(results_svm_orig["test_accuracy"].mean()),
                            "f1":       float(results_svm_orig["test_f1_weighted"].mean()),
                            "auc":      float(results_svm_orig["test_roc_auc"].mean())},
    "svm_augmented"     : {"accuracy": float(results_svm_aug["test_accuracy"].mean()),
                            "f1":       float(results_svm_aug["test_f1_weighted"].mean()),
                            "auc":      float(results_svm_aug["test_roc_auc"].mean())},
    "delta_accuracy_pct": float(delta_acc),
    "delta_auc_pct"     : float(delta_auc),
}
import json
with open("reports/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("✅ Résumé JSON sauvegardé : reports/final_summary.json")
print("\n🎉 Pipeline EEG end-to-end complet !")
